In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

here = Path.cwd()
candidates = [here, here.parent, here.parent.parent]
RAW = next((c / "data" / "raw" for c in candidates if (c / "data" / "raw").exists()), None)
if RAW is None:
    raise FileNotFoundError(f"Couldn't find data/raw from {here}")

PROCESSED = RAW.parent / "processed"
PROCESSED.mkdir(exist_ok=True)

print("Using:", RAW)

Using: c:\Users\user 1\Documents\ethiopia-fi-forecast\data\raw


In [4]:
df = pd.read_csv(RAW / "ethiopia_fi_unified_data.csv")
ref_codes = pd.read_csv(RAW / "reference_codes.csv")

print(df.shape)
print(df.columns.tolist())
df.head(10)

(43, 34)
['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.00,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.00,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.00,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.00,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.00,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
5,REC_0006,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,49.00,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,Account ownership increased from 46% to 49%,Survey Oct-Nov 2024,NaN
6,REC_0007,observation,NaN,ACCESS,Mobile Money Account Rate,ACC_MM_ACCOUNT,higher_better,4.70,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
7,REC_0008,observation,NaN,ACCESS,Mobile Money Account Rate,ACC_MM_ACCOUNT,higher_better,9.45,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Doubled from 2021,NaN
8,REC_0009,observation,NaN,ACCESS,4G Population Coverage,ACC_4G_COV,higher_better,37.50,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Before major expansion,NaN
9,REC_0010,observation,NaN,ACCESS,4G Population Coverage,ACC_4G_COV,higher_better,70.80,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,4G coverage doubled from 37.5% to 70.8%,Major infrastructure investment,NaN


In [5]:
print("--- record_type ---")
print(df["record_type"].value_counts())

print("\n--- pillar ---")
print(df["pillar"].value_counts(dropna=False))

print("\n--- category (events) ---")
print(df["category"].value_counts(dropna=False))

print("\n--- confidence ---")
print(df["confidence"].value_counts(dropna=False))

--- record_type ---
record_type
observation    30
event          10
target          3
Name: count, dtype: int64

--- pillar ---
pillar
ACCESS           16
USAGE            11
NaN              10
GENDER            5
AFFORDABILITY     1
Name: count, dtype: int64

--- category (events) ---
category
NaN               33
product_launch     2
infrastructure     2
policy             2
market_entry       1
milestone          1
partnership        1
pricing            1
Name: count, dtype: int64

--- confidence ---
confidence
high      40
medium     3
Name: count, dtype: int64


In [6]:
obs = df[df["record_type"] == "observation"].copy()
obs["observation_date"] = pd.to_datetime(obs["observation_date"], errors="coerce")

print(obs.groupby(["pillar", "indicator_code"])["value_numeric"].agg(["count", "min", "max"]))
print("\nDate range:", obs["observation_date"].min(), "to", obs["observation_date"].max())

                                  count           min           max
pillar        indicator_code                                       
ACCESS        ACC_4G_COV              2  3.750000e+01  7.080000e+01
              ACC_FAYDA               3  8.000000e+06  1.500000e+07
              ACC_MM_ACCOUNT          2  4.700000e+00  9.450000e+00
              ACC_MOBILE_PEN          1  6.140000e+01  6.140000e+01
              ACC_OWNERSHIP           6  2.200000e+01  5.600000e+01
AFFORDABILITY AFF_DATA_INCOME         1  2.000000e+00  2.000000e+00
GENDER        GEN_GAP_ACC             2  1.800000e+01  2.000000e+01
              GEN_GAP_MOBILE          1  2.400000e+01  2.400000e+01
              GEN_MM_SHARE            1  1.400000e+01  1.400000e+01
USAGE         USG_ACTIVE_RATE         1  6.600000e+01  6.600000e+01
              USG_ATM_COUNT           1  1.193000e+08  1.193000e+08
              USG_ATM_VALUE           1  1.561000e+11  1.561000e+11
              USG_CROSSOVER           1  1.08000

In [7]:
events = df[df["record_type"] == "event"].copy()
# adjust date/name columns below to match your real column list from Cell 2 if different
date_col = "observation_date" if "observation_date" in events.columns else "event_date"
events[date_col] = pd.to_datetime(events[date_col], errors="coerce")
events.sort_values(date_col)[["record_id", "indicator", "category", date_col]]

,record_id,indicator,category,observation_date
33,EVT_0001,Telebirr Launch,product_launch,2021-05-17
41,EVT_0009,NFIS-II Strategy Launch,policy,2021-09-01
34,EVT_0002,Safaricom Ethiopia Commercial Launch,market_entry,2022-08-01
35,EVT_0003,M-Pesa Ethiopia Launch,product_launch,2023-08-01
36,EVT_0004,Fayda Digital ID Program Rollout,infrastructure,2024-01-01
37,EVT_0005,Foreign Exchange Liberalization,policy,2024-07-29
38,EVT_0006,P2P Transaction Count Surpasses ATM,milestone,2024-10-01
39,EVT_0007,M-Pesa EthSwitch Integration,partnership,2025-10-27
42,EVT_0010,Safaricom Ethiopia Price Increase,pricing,2025-12-15
40,EVT_0008,EthioPay Instant Payment System Launch,infrastructure,2025-12-18


In [12]:
events = df[df["record_type"] == "event"].copy()
events["observation_date"] = pd.to_datetime(events["observation_date"], errors="coerce")
events.sort_values("observation_date")[["record_id", "indicator", "category", "observation_date"]]

,record_id,indicator,category,observation_date
33,EVT_0001,Telebirr Launch,product_launch,2021-05-17
41,EVT_0009,NFIS-II Strategy Launch,policy,2021-09-01
34,EVT_0002,Safaricom Ethiopia Commercial Launch,market_entry,2022-08-01
35,EVT_0003,M-Pesa Ethiopia Launch,product_launch,2023-08-01
36,EVT_0004,Fayda Digital ID Program Rollout,infrastructure,2024-01-01
37,EVT_0005,Foreign Exchange Liberalization,policy,2024-07-29
38,EVT_0006,P2P Transaction Count Surpasses ATM,milestone,2024-10-01
39,EVT_0007,M-Pesa EthSwitch Integration,partnership,2025-10-27
42,EVT_0010,Safaricom Ethiopia Price Increase,pricing,2025-12-15
40,EVT_0008,EthioPay Instant Payment System Launch,infrastructure,2025-12-18


In [13]:
links = df[df["record_type"] == "impact_link"]
links_view = links[["record_id", "indicator_code", "related_indicator", "relationship_type",
                     "impact_direction", "impact_magnitude", "impact_estimate", "lag_months",
                     "evidence_basis", "comparable_country", "confidence", "notes"]]
links_view

,record_id,indicator_code,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,confidence,notes


In [14]:
ref_codes[ref_codes["field_name"].str.contains("indicator_code|related_indicator|relationship", case=False, na=False)] if "field_name" in ref_codes.columns else ref_codes.head(20)

,field,code,description,applies_to
0,record_type,observation,Actual measured value from a source,All
1,record_type,event,Policy launch market event or milestone,All
2,record_type,impact_link,Relationship between event and indicator (link...,All
3,record_type,target,Policy target or official goal,All
4,record_type,baseline,Starting point for comparison,All
5,record_type,forecast,Predicted future value,All
6,category,product_launch,New product or service introduced,event
7,category,market_entry,New competitor enters market,event
8,category,market_exit,Competitor leaves market,event
9,category,policy,Government strategy or regulatory framework,event


In [15]:
ref_codes[ref_codes["field"] == "related_indicator"]
ref_codes[ref_codes["field"] == "relationship_type"]

,field,code,description,applies_to
55,relationship_type,direct,Event directly causes change in indicator,impact_link
56,relationship_type,indirect,Event causes change through intermediate mecha...,impact_link
57,relationship_type,enabling,Event creates conditions for future change,impact_link
58,relationship_type,constraining,Event limits potential improvement,impact_link


In [16]:
links = df[df["record_type"] == "impact_link"]
print(links["indicator_code"].unique())
print(links["related_indicator"].unique())

<ArrowStringArray>
[]
Length: 0, dtype: str
[]


In [17]:
ref_codes[ref_codes["field"] == "related_indicator"]
ref_codes[ref_codes["field"] == "indicator_code"]

,field,code,description,applies_to


In [18]:
pd.set_option("display.max_colwidth", None)
links.dropna(axis=1, how="all")

""


In [19]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
links = df[df["record_type"] == "impact_link"]
links.dropna(axis=1, how="all")

""


In [20]:
for col in links.columns:
    non_null = links[col].dropna()
    if len(non_null) > 0:
        print(f"--- {col} ---")
        print(non_null.unique()[:10])
        print()

In [21]:
print(len(links))
print(df["record_type"].value_counts())

0
record_type
observation    30
event          10
target          3
Name: count, dtype: int64


In [22]:
print(df["indicator_code"].dropna().unique())
print(df["category"].dropna().unique())

<ArrowStringArray>
[     'ACC_OWNERSHIP',     'ACC_MM_ACCOUNT',         'ACC_4G_COV',
     'ACC_MOBILE_PEN',          'ACC_FAYDA',      'USG_P2P_COUNT',
      'USG_P2P_VALUE',      'USG_ATM_COUNT',      'USG_ATM_VALUE',
      'USG_CROSSOVER', 'USG_TELEBIRR_USERS', 'USG_TELEBIRR_VALUE',
    'USG_MPESA_USERS',   'USG_MPESA_ACTIVE',    'USG_ACTIVE_RATE',
    'AFF_DATA_INCOME',        'GEN_GAP_ACC',       'GEN_MM_SHARE',
     'GEN_GAP_MOBILE',       'EVT_TELEBIRR',      'EVT_SAFARICOM',
          'EVT_MPESA',          'EVT_FAYDA',      'EVT_FX_REFORM',
      'EVT_CROSSOVER',  'EVT_MPESA_INTEROP',       'EVT_ETHIOPAY',
          'EVT_NFIS2',   'EVT_SAFCOM_PRICE']
Length: 29, dtype: str
<ArrowStringArray>
['product_launch',   'market_entry', 'infrastructure',         'policy',
      'milestone',    'partnership',        'pricing']
Length: 7, dtype: str


In [23]:
events = df[df["record_type"] == "event"].copy()
events["observation_date"] = pd.to_datetime(events["observation_date"])
events.sort_values("observation_date")[
    ["record_id", "indicator", "category", "observation_date", "value_text", "notes"]
]

,record_id,indicator,category,observation_date,value_text,notes
33,EVT_0001,Telebirr Launch,product_launch,2021-05-17,Launched,NaN
41,EVT_0009,NFIS-II Strategy Launch,policy,2021-09-01,Launched,NaN
34,EVT_0002,Safaricom Ethiopia Commercial Launch,market_entry,2022-08-01,Launched,NaN
35,EVT_0003,M-Pesa Ethiopia Launch,product_launch,2023-08-01,Launched,NaN
36,EVT_0004,Fayda Digital ID Program Rollout,infrastructure,2024-01-01,Launched,NaN
37,EVT_0005,Foreign Exchange Liberalization,policy,2024-07-29,Implemented,NaN
38,EVT_0006,P2P Transaction Count Surpasses ATM,milestone,2024-10-01,Achieved,NaN
39,EVT_0007,M-Pesa EthSwitch Integration,partnership,2025-10-27,Launched,NaN
42,EVT_0010,Safaricom Ethiopia Price Increase,pricing,2025-12-15,Implemented,NaN
40,EVT_0008,EthioPay Instant Payment System Launch,infrastructure,2025-12-18,Launched,NaN


In [24]:
new_impact_links = [
    {
        "record_id": "LINK_0001",
        "record_type": "impact_link",
        "parent_id": "EVT_0001",              # Telebirr launch
        "pillar": "ACCESS",
        "related_indicator": "ACC_MM_ACCOUNT",
        "relationship_type": "direct",
        "impact_direction": "increase",
        "impact_magnitude": "high",
        "impact_estimate": 4.7,                 # pp change observed 2021 baseline
        "lag_months": 6,
        "evidence_basis": "empirical",
        "comparable_country": None,
        "confidence": "high",
        "collected_by": "<your name>",
        "collection_date": "2026-07-18",
        "notes": "Telebirr launched May 2021; mobile money account rate was 4.7% by end 2021 survey."
    },
    {
        "record_id": "LINK_0002",
        "record_type": "impact_link",
        "parent_id": "EVT_0003",              # M-Pesa Ethiopia launch
        "pillar": "USAGE",
        "related_indicator": "USG_MPESA_USERS",
        "relationship_type": "direct",
        "impact_direction": "increase",
        "impact_magnitude": "high",
        "impact_estimate": None,
        "lag_months": 12,
        "evidence_basis": "empirical",
        "comparable_country": "Kenya",
        "confidence": "medium",
        "collected_by": "<your name>",
        "collection_date": "2026-07-18",
        "notes": "M-Pesa Ethiopia reached 10.8M users by end-2024, comparable ramp to Kenya's early M-Pesa growth."
    },
    {
        "record_id": "LINK_0003",
        "record_type": "impact_link",
        "parent_id": "EVT_0002",              # Safaricom market entry
        "pillar": "ACCESS",
        "related_indicator": "ACC_MOBILE_PEN",
        "relationship_type": "enabling",
        "impact_direction": "increase",
        "impact_magnitude": "medium",
        "impact_estimate": None,
        "lag_months": 12,
        "evidence_basis": "theoretical",
        "comparable_country": None,
        "confidence": "low",
        "collected_by": "<your name>",
        "collection_date": "2026-07-18",
        "notes": "End of state telecom monopoly plausibly increased competition and mobile penetration."
    },
    # ... repeat for remaining events: EVT_0004 (Fayda) -> ACC_FAYDA,
    # EVT_0005 (FX liberalization) -> possibly AFF_DATA_INCOME or USAGE indicators,
    # EVT_0007 (M-Pesa/EthSwitch interop) -> USG_MPESA_ACTIVE,
    # EVT_0008 (EthioPay) -> USG_CROSSOVER,
    # EVT_0009 (NFIS-II) -> ACC_OWNERSHIP (policy target driver),
    # EVT_0010 (Safaricom price increase) -> ACC_MOBILE_PEN, impact_direction="decrease"
]

df_enriched = pd.concat([df, pd.DataFrame(new_impact_links)], ignore_index=True)
print(df_enriched["record_type"].value_counts())

record_type
observation    30
event          10
target          3
impact_link     3
Name: count, dtype: int64


In [26]:
new_impact_links = [
    {
        "record_id": "LINK_0004",
        "record_type": "impact_link",
        "parent_id": "EVT_0001",
        "pillar": "ACCESS",
        "related_indicator": "ACC_OWNERSHIP",
        "relationship_type": "indirect",
        "impact_direction": "increase",
        "impact_magnitude": "medium",
        "impact_estimate": None,
        "lag_months": 24,
        "evidence_basis": "empirical",
        "comparable_country": None,
        "confidence": "medium",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "Overall account ownership rose 46%->49% (2021-24); Telebirr likely a partial contributor alongside other drivers, given most mobile money users already had bank accounts."
    },
    {
        "record_id": "LINK_0005",
        "record_type": "impact_link",
        "parent_id": "EVT_0009",
        "pillar": "ACCESS",
        "related_indicator": "ACC_OWNERSHIP",
        "relationship_type": "enabling",
        "impact_direction": "increase",
        "impact_magnitude": "medium",
        "impact_estimate": None,
        "lag_months": 12,
        "evidence_basis": "expert",
        "comparable_country": None,
        "confidence": "medium",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "NFIS-II sets the 70% account ownership target and coordinates NBE/operator programs; enabling rather than a direct causal driver."
    },
    {
        "record_id": "LINK_0006",
        "record_type": "impact_link",
        "parent_id": "EVT_0009",
        "pillar": "ACCESS",
        "related_indicator": "ACC_FAYDA",
        "relationship_type": "enabling",
        "impact_direction": "increase",
        "impact_magnitude": "medium",
        "impact_estimate": None,
        "lag_months": 12,
        "evidence_basis": "expert",
        "comparable_country": None,
        "confidence": "low",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "NFIS-II strategy likely coordinated policy priority behind the Fayda digital ID rollout."
    },
    {
        "record_id": "LINK_0007",
        "record_type": "impact_link",
        "parent_id": "EVT_0004",
        "pillar": "ACCESS",
        "related_indicator": "ACC_OWNERSHIP",
        "relationship_type": "enabling",
        "impact_direction": "increase",
        "impact_magnitude": "medium",
        "impact_estimate": None,
        "lag_months": 18,
        "evidence_basis": "theoretical",
        "comparable_country": None,
        "confidence": "low",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "Digital ID can ease KYC/identity requirements for account opening; effect not yet visible in Findex data (next survey needed)."
    },
    {
        "record_id": "LINK_0008",
        "record_type": "impact_link",
        "parent_id": "EVT_0004",
        "pillar": "GENDER",
        "related_indicator": "GEN_GAP_ACC",
        "relationship_type": "indirect",
        "impact_direction": "decrease",
        "impact_magnitude": "low",
        "impact_estimate": None,
        "lag_months": 24,
        "evidence_basis": "theoretical",
        "comparable_country": None,
        "confidence": "low",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "Digital ID may disproportionately help women lacking formal documentation, potentially narrowing the account ownership gender gap (lower_better indicator)."
    },
    {
        "record_id": "LINK_0009",
        "record_type": "impact_link",
        "parent_id": "EVT_0005",
        "pillar": "AFFORDABILITY",
        "related_indicator": "AFF_DATA_INCOME",
        "relationship_type": "indirect",
        "impact_direction": "increase",
        "impact_magnitude": "medium",
        "impact_estimate": None,
        "lag_months": 6,
        "evidence_basis": "theoretical",
        "comparable_country": None,
        "confidence": "low",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "Birr float/depreciation could raise imported device and data costs in the short term, worsening data affordability (lower_better indicator)."
    },
    {
        "record_id": "LINK_0010",
        "record_type": "impact_link",
        "parent_id": "EVT_0005",
        "pillar": "USAGE",
        "related_indicator": "USG_TELEBIRR_VALUE",
        "relationship_type": "indirect",
        "impact_direction": "increase",
        "impact_magnitude": "medium",
        "impact_estimate": None,
        "lag_months": 6,
        "evidence_basis": "theoretical",
        "comparable_country": None,
        "confidence": "medium",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "Currency depreciation inflates nominal ETB transaction values, independent of real usage growth — a confound to flag when interpreting value-based indicators."
    },
    {
        "record_id": "LINK_0011",
        "record_type": "impact_link",
        "parent_id": "EVT_0006",
        "pillar": "USAGE",
        "related_indicator": "USG_ATM_COUNT",
        "relationship_type": "constraining",
        "impact_direction": "stabilize",
        "impact_magnitude": "low",
        "impact_estimate": None,
        "lag_months": 12,
        "evidence_basis": "empirical",
        "comparable_country": None,
        "confidence": "medium",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "P2P overtaking ATM (Oct 2024) is itself an outcome milestone; going forward it may cap further ATM transaction growth as digital rails absorb demand."
    },
    {
        "record_id": "LINK_0012",
        "record_type": "impact_link",
        "parent_id": "EVT_0007",
        "pillar": "USAGE",
        "related_indicator": "USG_MPESA_ACTIVE",
        "relationship_type": "direct",
        "impact_direction": "increase",
        "impact_magnitude": "medium",
        "impact_estimate": None,
        "lag_months": 6,
        "evidence_basis": "theoretical",
        "comparable_country": "Kenya",
        "confidence": "medium",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "EthSwitch interoperability should raise M-Pesa's 90-day active-user rate by widening where funds can be sent/received."
    },
    {
        "record_id": "LINK_0013",
        "record_type": "impact_link",
        "parent_id": "EVT_0008",
        "pillar": "USAGE",
        "related_indicator": "USG_CROSSOVER",
        "relationship_type": "direct",
        "impact_direction": "increase",
        "impact_magnitude": "high",
        "impact_estimate": None,
        "lag_months": 12,
        "evidence_basis": "theoretical",
        "comparable_country": None,
        "confidence": "medium",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "EthioPay real-time payment rail expected to further accelerate the P2P/ATM crossover ratio beyond the initial 1.08 milestone."
    },
    {
        "record_id": "LINK_0014",
        "record_type": "impact_link",
        "parent_id": "EVT_0010",
        "pillar": "ACCESS",
        "related_indicator": "ACC_MOBILE_PEN",
        "relationship_type": "constraining",
        "impact_direction": "decrease",
        "impact_magnitude": "low",
        "impact_estimate": None,
        "lag_months": 6,
        "evidence_basis": "theoretical",
        "comparable_country": None,
        "confidence": "medium",
        "collected_by": "<your name>", "collection_date": "2026-07-18",
        "notes": "20-82% price increase on data/voice may slow mobile subscription growth among price-sensitive users, indirectly constraining mobile-money-enabled account growth."
    },
]

df_enriched = pd.concat([df, pd.DataFrame(new_impact_links)], ignore_index=True)
print(df_enriched["record_type"].value_counts())   # should show impact_link: 14

record_type
observation    30
impact_link    11
event          10
target          3
Name: count, dtype: int64


In [27]:
out_path = PROCESSED / "ethiopia_fi_unified_data_enriched.csv"
df_enriched.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: c:\Users\user 1\Documents\ethiopia-fi-forecast\data\processed\ethiopia_fi_unified_data_enriched.csv


In [28]:
new_observations = [
    {
        "record_id": "OBS_NEW_01",
        "record_type": "observation",
        "pillar": "ACCESS",
        "indicator": "Account Ownership Rate",
        "indicator_code": "ACC_OWNERSHIP",
        "indicator_direction": "higher_better",
        "value_numeric": 55,          # <-- replace with real figure
        "value_type": "percentage",
        "unit": "%",
        "observation_date": "2021-12-31",
        "fiscal_year": 2021,
        "gender": "all",
        "location": "urban",          # urban vs rural disaggregation
        "source_name": "Global Findex 2021",
        "source_type": "survey",
        "source_url": "https://www.worldbank.org/en/publication/globalfindex",
        "confidence": "high",
        "collected_by": "<your name>",
        "collection_date": "2026-07-18",
        "notes": "Urban account ownership for Task 2 urban vs rural analysis."
    },
    {
        "record_id": "OBS_NEW_02",
        "record_type": "observation",
        "pillar": "ACCESS",
        "indicator": "Account Ownership Rate",
        "indicator_code": "ACC_OWNERSHIP",
        "indicator_direction": "higher_better",
        "value_numeric": 40,          # <-- replace with real figure
        "value_type": "percentage",
        "unit": "%",
        "observation_date": "2021-12-31",
        "fiscal_year": 2021,
        "gender": "all",
        "location": "rural",
        "source_name": "Global Findex 2021",
        "source_type": "survey",
        "source_url": "https://www.worldbank.org/en/publication/globalfindex",
        "confidence": "high",
        "collected_by": "<your name>",
        "collection_date": "2026-07-18",
        "notes": "Rural account ownership for Task 2 urban vs rural analysis."
    },
]

df_enriched = pd.concat([df_enriched, pd.DataFrame(new_observations)], ignore_index=True)
df_enriched.to_csv(PROCESSED / "ethiopia_fi_unified_data_enriched.csv", index=False)
print(df_enriched["record_type"].value_counts())

record_type
observation    32
impact_link    11
event          10
target          3
Name: count, dtype: int64
